# PropIQ — Week 4: Source-to-Bronze Ingestion

**Project:** P15 PropIQ — Real Estate Market Analytics
**Notebook:** `notebooks/02_bronze_ingestion.ipynb`
**Team:** Ms. Thota Madhulika, Ms. P. Lakshmi Naga Sree, Ms. Vadlamuru Rishitha

## 3.1 Week-4 objective

This notebook moves every **approved batch source file** for PropIQ into a
**persistent Bronze Delta table**, while preserving the original source
business values exactly as received.

Bronze does **not** clean, cast, deduplicate, or reinterpret any business
value. Bronze only adds controlled ingestion and lineage metadata on top of
the untouched source columns.

Streaming files (`listing_status_event_drop_*.json`) are **excluded** from
this notebook. Per the Data Pack manifest they are first used in Week 10 and
belong to the streaming-simulation phase, not controlled batch ingestion.


## 3.2 Environment setup

PropIQ uses the same catalog, schema and Volume confirmed during Week 3
exploration. This notebook does **not** create a new schema — it only
selects the existing one and verifies the selection.

| Item | Value |
|---|---|
| Catalog | `workspace` |
| Schema | `default` |
| Unity Catalog Volume | `/Volumes/workspace/default/propiq/` |


In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT
  current_catalog() AS active_catalog,
  current_schema()  AS active_schema;

active_catalog,active_schema
workspace,default


**Expected observation:** the result must show `workspace` and `default`.
If it does not, stop and correct the catalog/schema selection before
continuing — every unqualified table name created below resolves inside this
location.

### Define the controlled Week-4 run

Every Bronze table built in this notebook is stamped with the same
`ingestion_run_id` and `schema_version` for this run. Keep these values
unchanged while testing and during the rerun proof in Part 6; change them
only when a genuinely new, mentor-approved batch is loaded.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW week4_run_control AS
SELECT
  'W04_PROPIQ_RUN01'    AS ingestion_run_id,
  'propiq_source_v1.0'  AS schema_version;

SELECT * FROM week4_run_control;


ingestion_run_id,schema_version
W04_PROPIQ_RUN01,propiq_source_v1.0


## 3.3 Source inventory

Four approved batch source files are in scope for Week 4. These are the same
four confirmed in the Week-2 Data Dictionary and the Data Pack
`source_manifest.csv`. The six `listing_status_event_drop_*.json` streaming
drops are explicitly excluded (first-use Week 10).

| # | Source file | Format | Volume path | Business key | Bronze target table |
|---|---|---|---|---|---|
| 1 | `listings.parquet`  | Parquet    | `/Volumes/workspace/default/propiq/listings.parquet`  | `listing_id`  (physical: `record_uid`) | `bronze_propiq_listings` |
| 2 | `leads.csv`         | CSV        | `/Volumes/workspace/default/propiq/leads.csv`         | `lead_id`     (physical: `record_uid`) | `bronze_propiq_leads` |
| 3 | `localities.json`   | JSON Lines | `/Volumes/workspace/default/propiq/localities.json`   | `locality_id` | `bronze_propiq_localities` |
| 4 | `brokers.csv`       | CSV        | `/Volumes/workspace/default/propiq/brokers.csv`       | `broker_id`   | `bronze_propiq_brokers` |

**Minimum Bronze metadata added to every table:** `_source_file_name`,
`_source_file_path`, `_ingested_at`, `_ingestion_run_id`, `_schema_version`,
`_record_hash`.

`_rescued_payload` is added for the CSV and JSON sources because their
readers run in `PERMISSIVE` mode with a declared corrupt-record column. The
standard Databricks Parquet reader used for `listings.parquet` does not
produce an equivalent corrupt-record column, so `_rescued_payload` is not
applicable there and is intentionally omitted for that source.


## 3.4 File check

Confirm that every approved batch file is visible in the Volume before any
ingestion cell runs.

In [0]:
%fs
ls /Volumes/workspace/default/propiq


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/propiq/brokers.csv,brokers.csv,40428,1784738115000
dbfs:/Volumes/workspace/default/propiq/leads.csv,leads.csv,17096212,1784738237000
dbfs:/Volumes/workspace/default/propiq/listings.parquet,listings.parquet,2533387,1784738145000
dbfs:/Volumes/workspace/default/propiq/localities.json,localities.json,24420,1784738115000


**Expected observation:** the listing must include all four files —
`listings.parquet`, `leads.csv`, `localities.json`, `brokers.csv`. If any file
is missing, upload it to the Volume before continuing; do not proceed with a
partial source set.

# Part 3 — Build `bronze_propiq_listings`

## 3.1 Identify the source

| Item | Value |
|---|---|
| Source file | `listings.parquet` |
| Format | Parquet |
| Volume path | `/Volumes/workspace/default/propiq/listings.parquet` |
| Target Bronze table | `bronze_propiq_listings` |



### Read the source (standard Parquet reader)

`spark.read.parquet` is the standard Databricks reader for Parquet. Parquet
is self-describing, so no explicit schema is declared here — the file's own
embedded schema is used as-is, and every business column is preserved
unmodified.

In [0]:
# Read listings.parquet using the standard Parquet reader
listings_df = spark.read.parquet("/Volumes/workspace/default/propiq/listings.parquet")

# Make the DataFrame available to Spark SQL
listings_df.createOrReplaceTempView("listings_source")


## 3.2 Inspect the source

In [0]:
%sql
SELECT * FROM listings_source LIMIT 10;


record_uid,listing_id,locality_id,broker_id,property_type,bedrooms,furnishing,built_up_area_sqft,asking_price_inr,price_per_sqft,listing_created_date,last_updated_timestamp,completion_date,listing_status,source_system,source_record_id,batch_id
LISTING-PHY-0000001,LST-0000001,LOC-062,BRK-0191,Apartment,4,Furnished,1631,11005988,6748,2025-05-15,2025-09-03T21:21:36,null,active,BROKER_CRM,SRC-L-0000001,BATCH-2026-01
LISTING-PHY-0000002,LST-0000002,LOC-056,BRK-0130,Apartment,2,Semi-furnished,1157,21793252,18836,2025-07-17,2025-08-10T14:05:46,null,active,BROKER_CRM,SRC-L-0000002,BATCH-2026-01
LISTING-PHY-0000003,LST-0000003,LOC-045,BRK-0215,Apartment,1,Semi-furnished,1973,16873096,8552,2024-12-23,2025-01-02T13:35:06,2025-03-01,rented,AGENCY_UPLOAD,SRC-L-0000003,BATCH-2026-01
LISTING-PHY-0000004,LST-0000004,LOC-036,BRK-0177,Apartment,4,Furnished,1164,21144060,18165,2024-10-31,2024-11-08T16:45:22,null,paused,PORTAL_FEED_A,SRC-L-0000004,BATCH-2026-01
LISTING-PHY-0000005,LST-0000005,LOC-041,BRK-0298,Apartment,3,Furnished,1478,9571528,6476,2025-07-28,2025-10-17T21:50:45,2025-12-04,sold,BROKER_CRM,SRC-L-0000005,BATCH-2026-01
LISTING-PHY-0000006,LST-0000006,LOC-014,BRK-0050,Apartment,3,Unfurnished,1827,10529001,5763,2024-05-25,2024-08-28T03:01:36,2024-09-26,sold,PORTAL_FEED_A,SRC-L-0000006,BATCH-2026-01
LISTING-PHY-0000007,LST-0000007,LOC-029,BRK-0055,Villa,4,Semi-furnished,4740,34199100,7215,2025-10-02,2025-11-22T06:51:33,null,expired,PORTAL_FEED_A,SRC-L-0000007,BATCH-2026-01
LISTING-PHY-0000008,LST-0000008,LOC-074,BRK-0011,Apartment,1,Semi-furnished,1296,12205728,9418,2024-01-30,2024-05-18T02:03:22,null,active,AGENCY_UPLOAD,SRC-L-0000008,BATCH-2026-01
LISTING-PHY-0000009,LST-0000009,LOC-046,BRK-0017,Apartment,1,Unfurnished,1301,9395822,7222,2025-02-04,2025-06-11T17:21:51,2025-06-09,rented,BROKER_CRM,SRC-L-0000009,BATCH-2026-01
LISTING-PHY-0000010,LST-0000010,LOC-071,BRK-0201,Villa,2,Semi-furnished,3882,39615810,10205,2024-02-12,2024-04-30T07:39:38,2024-05-01,sold,PORTAL_FEED_A,SRC-L-0000010,BATCH-2026-01


Confirm the approved business columns are present with no unexpected merged or missing fields.

In [0]:
%sql
DESCRIBE listings_source;


col_name,data_type,comment
record_uid,string,null
listing_id,string,null
locality_id,string,null
broker_id,string,null
property_type,string,null
bedrooms,bigint,null
furnishing,string,null
built_up_area_sqft,bigint,null
asking_price_inr,bigint,null
price_per_sqft,bigint,null


In [0]:
%sql
SELECT COUNT(*) AS listings_source_count
FROM listings_source;


listings_source_count
50200


Record the displayed source count in the Week-4 log before continuing.

## 3.3 Create Bronze — add metadata

`listings_bronze_ready` keeps every approved business column untouched and adds:

- `_source_file_name`, `_source_file_path` — where the row came from;
- `_ingested_at` — when the row entered Bronze;
- `_ingestion_run_id`, `_schema_version` — which controlled run and source
  contract produced this row;
- `_record_hash` — a repeatable fingerprint built from the business columns
  in a fixed order, so any future silent value change can be detected;



In [0]:
%sql
CREATE OR REPLACE TEMP VIEW listings_bronze_ready AS
SELECT
  s.*,
  'listings.parquet' AS _source_file_name,
  '/Volumes/workspace/default/propiq/listings.parquet' AS _source_file_path,
  current_timestamp() AS _ingested_at,
  r.ingestion_run_id AS _ingestion_run_id,
  r.schema_version AS _schema_version,
  sha2(concat_ws('||',
    coalesce(cast(s.record_uid AS STRING), '<NULL>'),
    coalesce(cast(s.listing_id AS STRING), '<NULL>'),
    coalesce(cast(s.locality_id AS STRING), '<NULL>'),
    coalesce(cast(s.broker_id AS STRING), '<NULL>'),
    coalesce(cast(s.property_type AS STRING), '<NULL>'),
    coalesce(cast(s.bedrooms AS STRING), '<NULL>'),
    coalesce(cast(s.built_up_area_sqft AS STRING), '<NULL>'),
    coalesce(cast(s.asking_price_inr AS STRING), '<NULL>'),
    coalesce(cast(s.price_per_sqft AS STRING), '<NULL>'),
    coalesce(cast(s.listing_status AS STRING), '<NULL>'),
    coalesce(cast(s.listing_created_date AS STRING), '<NULL>')
  ), 256) AS _record_hash
FROM listings_source s
CROSS JOIN week4_run_control r;


In [0]:
%sql
SELECT * FROM listings_bronze_ready LIMIT 10;


record_uid,listing_id,locality_id,broker_id,property_type,bedrooms,furnishing,built_up_area_sqft,asking_price_inr,price_per_sqft,listing_created_date,last_updated_timestamp,completion_date,listing_status,source_system,source_record_id,batch_id,_source_file_name,_source_file_path,_ingested_at,_ingestion_run_id,_schema_version,_record_hash
LISTING-PHY-0000001,LST-0000001,LOC-062,BRK-0191,Apartment,4,Furnished,1631,11005988,6748,2025-05-15,2025-09-03T21:21:36,null,active,BROKER_CRM,SRC-L-0000001,BATCH-2026-01,listings.parquet,/Volumes/workspace/default/propiq/listings.parquet,2026-07-30T14:04:03.391Z,W04_PROPIQ_RUN01,propiq_source_v1.0,992ca31fa2edf5153a0e66a4d210ec7d181f2b986353f7168d7088ca9965a0dd
LISTING-PHY-0000002,LST-0000002,LOC-056,BRK-0130,Apartment,2,Semi-furnished,1157,21793252,18836,2025-07-17,2025-08-10T14:05:46,null,active,BROKER_CRM,SRC-L-0000002,BATCH-2026-01,listings.parquet,/Volumes/workspace/default/propiq/listings.parquet,2026-07-30T14:04:03.391Z,W04_PROPIQ_RUN01,propiq_source_v1.0,843c8fed6f0b32b10847443a0c8cb9081016ef086074c240ba826dcd8fc9d4e6
LISTING-PHY-0000003,LST-0000003,LOC-045,BRK-0215,Apartment,1,Semi-furnished,1973,16873096,8552,2024-12-23,2025-01-02T13:35:06,2025-03-01,rented,AGENCY_UPLOAD,SRC-L-0000003,BATCH-2026-01,listings.parquet,/Volumes/workspace/default/propiq/listings.parquet,2026-07-30T14:04:03.391Z,W04_PROPIQ_RUN01,propiq_source_v1.0,6b6876084e40223a8219c7d8acb36841c7d1efd310efe0f5b866fb146173a58c
LISTING-PHY-0000004,LST-0000004,LOC-036,BRK-0177,Apartment,4,Furnished,1164,21144060,18165,2024-10-31,2024-11-08T16:45:22,null,paused,PORTAL_FEED_A,SRC-L-0000004,BATCH-2026-01,listings.parquet,/Volumes/workspace/default/propiq/listings.parquet,2026-07-30T14:04:03.391Z,W04_PROPIQ_RUN01,propiq_source_v1.0,c2709284c2a539ca3091b2535828028ca3d93d280eeb019661e09d8119e2864e
LISTING-PHY-0000005,LST-0000005,LOC-041,BRK-0298,Apartment,3,Furnished,1478,9571528,6476,2025-07-28,2025-10-17T21:50:45,2025-12-04,sold,BROKER_CRM,SRC-L-0000005,BATCH-2026-01,listings.parquet,/Volumes/workspace/default/propiq/listings.parquet,2026-07-30T14:04:03.391Z,W04_PROPIQ_RUN01,propiq_source_v1.0,153680f8597247c187605ac647808b5e3ff04ded33cde9ba3b5b9f13b3982e05
LISTING-PHY-0000006,LST-0000006,LOC-014,BRK-0050,Apartment,3,Unfurnished,1827,10529001,5763,2024-05-25,2024-08-28T03:01:36,2024-09-26,sold,PORTAL_FEED_A,SRC-L-0000006,BATCH-2026-01,listings.parquet,/Volumes/workspace/default/propiq/listings.parquet,2026-07-30T14:04:03.391Z,W04_PROPIQ_RUN01,propiq_source_v1.0,8209854c956832dcd9582bef5a412714e714f6d25af547b7fb62bee92b4e3719
LISTING-PHY-0000007,LST-0000007,LOC-029,BRK-0055,Villa,4,Semi-furnished,4740,34199100,7215,2025-10-02,2025-11-22T06:51:33,null,expired,PORTAL_FEED_A,SRC-L-0000007,BATCH-2026-01,listings.parquet,/Volumes/workspace/default/propiq/listings.parquet,2026-07-30T14:04:03.391Z,W04_PROPIQ_RUN01,propiq_source_v1.0,e50579cacaaa1a1862fda5a19419cce9311b384156fba63894773c98fae304e5
LISTING-PHY-0000008,LST-0000008,LOC-074,BRK-0011,Apartment,1,Semi-furnished,1296,12205728,9418,2024-01-30,2024-05-18T02:03:22,null,active,AGENCY_UPLOAD,SRC-L-0000008,BATCH-2026-01,listings.parquet,/Volumes/workspace/default/propiq/listings.parquet,2026-07-30T14:04:03.391Z,W04_PROPIQ_RUN01,propiq_source_v1.0,395ba5bcdc091bf16bba3285fa5f0291175edc2e3a6583a82d3b2b1f14a72940
LISTING-PHY-0000009,LST-0000009,LOC-046,BRK-0017,Apartment,1,Unfurnished,1301,9395822,7222,2025-02-04,2025-06-11T17:21:51,2025-06-09,rented,BROKER_CRM,SRC-L-0000009,BATCH-2026-01,listings.parquet,/Volumes/workspace/default/propiq/listings.parquet,2026-07-30T14:04:03.391Z,W04_PROPIQ_RUN01,propiq_source_v1.0,f6053873890db17304c3ae71ff571ff8ef8cc4b9362e61a2880f67c662b0a08b
LISTING-PHY-0000010,LST-0000010,LOC-071,BRK-0201,Villa,2,Semi-furnished,3882,39615810,10205,2024-02-12,2024-04-30T07:39:38,2024-05-01,sold,PORTAL_FEED_A,SRC-L-0000010,BATCH-2026-01,listings.parquet,/Volumes/workspace/default/propiq/listings.parquet,2026-07-30T14:04:03.391Z,W04_PROPIQ_RUN01,propiq_source_v1.0,

### Persist `bronze_propiq_listings` as Delta

`CREATE OR REPLACE TABLE ... USING DELTA` writes the complete controlled
snapshot as a persistent Unity Catalog Delta table, supporting the
controlled full-refresh / safe-rerun pattern used in Week 4.

In [0]:
%sql
CREATE OR REPLACE TABLE bronze_propiq_listings
USING DELTA
AS
SELECT * FROM listings_bronze_ready;


num_affected_rows,num_inserted_rows


## 3.4 Verify the table

In [0]:
%sql
DESCRIBE DETAIL bronze_propiq_listings;


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,7f96fdf4-1aad-4f12-bdd8-746daf92bc85,workspace.default.bronze_propiq_listings,null,,2026-07-30T14:04:07.186Z,2026-07-30T14:04:14.000Z,List(),List(),1,2680273,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE TABLE bronze_propiq_listings;


col_name,data_type,comment
record_uid,string,null
listing_id,string,null
locality_id,string,null
broker_id,string,null
property_type,string,null
bedrooms,bigint,null
furnishing,string,null
built_up_area_sqft,bigint,null
asking_price_inr,bigint,null
price_per_sqft,bigint,null


In [0]:
%sql
SELECT * FROM bronze_propiq_listings LIMIT 10;


record_uid,listing_id,locality_id,broker_id,property_type,bedrooms,furnishing,built_up_area_sqft,asking_price_inr,price_per_sqft,listing_created_date,last_updated_timestamp,completion_date,listing_status,source_system,source_record_id,batch_id,_source_file_name,_source_file_path,_ingested_at,_ingestion_run_id,_schema_version,_record_hash
LISTING-PHY-0000001,LST-0000001,LOC-062,BRK-0191,Apartment,4,Furnished,1631,11005988,6748,2025-05-15,2025-09-03T21:21:36,null,active,BROKER_CRM,SRC-L-0000001,BATCH-2026-01,listings.parquet,/Volumes/workspace/default/propiq/listings.parquet,2026-07-30T14:04:08.257Z,W04_PROPIQ_RUN01,propiq_source_v1.0,992ca31fa2edf5153a0e66a4d210ec7d181f2b986353f7168d7088ca9965a0dd
LISTING-PHY-0000002,LST-0000002,LOC-056,BRK-0130,Apartment,2,Semi-furnished,1157,21793252,18836,2025-07-17,2025-08-10T14:05:46,null,active,BROKER_CRM,SRC-L-0000002,BATCH-2026-01,listings.parquet,/Volumes/workspace/default/propiq/listings.parquet,2026-07-30T14:04:08.257Z,W04_PROPIQ_RUN01,propiq_source_v1.0,843c8fed6f0b32b10847443a0c8cb9081016ef086074c240ba826dcd8fc9d4e6
LISTING-PHY-0000003,LST-0000003,LOC-045,BRK-0215,Apartment,1,Semi-furnished,1973,16873096,8552,2024-12-23,2025-01-02T13:35:06,2025-03-01,rented,AGENCY_UPLOAD,SRC-L-0000003,BATCH-2026-01,listings.parquet,/Volumes/workspace/default/propiq/listings.parquet,2026-07-30T14:04:08.257Z,W04_PROPIQ_RUN01,propiq_source_v1.0,6b6876084e40223a8219c7d8acb36841c7d1efd310efe0f5b866fb146173a58c
LISTING-PHY-0000004,LST-0000004,LOC-036,BRK-0177,Apartment,4,Furnished,1164,21144060,18165,2024-10-31,2024-11-08T16:45:22,null,paused,PORTAL_FEED_A,SRC-L-0000004,BATCH-2026-01,listings.parquet,/Volumes/workspace/default/propiq/listings.parquet,2026-07-30T14:04:08.257Z,W04_PROPIQ_RUN01,propiq_source_v1.0,c2709284c2a539ca3091b2535828028ca3d93d280eeb019661e09d8119e2864e
LISTING-PHY-0000005,LST-0000005,LOC-041,BRK-0298,Apartment,3,Furnished,1478,9571528,6476,2025-07-28,2025-10-17T21:50:45,2025-12-04,sold,BROKER_CRM,SRC-L-0000005,BATCH-2026-01,listings.parquet,/Volumes/workspace/default/propiq/listings.parquet,2026-07-30T14:04:08.257Z,W04_PROPIQ_RUN01,propiq_source_v1.0,153680f8597247c187605ac647808b5e3ff04ded33cde9ba3b5b9f13b3982e05
LISTING-PHY-0000006,LST-0000006,LOC-014,BRK-0050,Apartment,3,Unfurnished,1827,10529001,5763,2024-05-25,2024-08-28T03:01:36,2024-09-26,sold,PORTAL_FEED_A,SRC-L-0000006,BATCH-2026-01,listings.parquet,/Volumes/workspace/default/propiq/listings.parquet,2026-07-30T14:04:08.257Z,W04_PROPIQ_RUN01,propiq_source_v1.0,8209854c956832dcd9582bef5a412714e714f6d25af547b7fb62bee92b4e3719
LISTING-PHY-0000007,LST-0000007,LOC-029,BRK-0055,Villa,4,Semi-furnished,4740,34199100,7215,2025-10-02,2025-11-22T06:51:33,null,expired,PORTAL_FEED_A,SRC-L-0000007,BATCH-2026-01,listings.parquet,/Volumes/workspace/default/propiq/listings.parquet,2026-07-30T14:04:08.257Z,W04_PROPIQ_RUN01,propiq_source_v1.0,e50579cacaaa1a1862fda5a19419cce9311b384156fba63894773c98fae304e5
LISTING-PHY-0000008,LST-0000008,LOC-074,BRK-0011,Apartment,1,Semi-furnished,1296,12205728,9418,2024-01-30,2024-05-18T02:03:22,null,active,AGENCY_UPLOAD,SRC-L-0000008,BATCH-2026-01,listings.parquet,/Volumes/workspace/default/propiq/listings.parquet,2026-07-30T14:04:08.257Z,W04_PROPIQ_RUN01,propiq_source_v1.0,395ba5bcdc091bf16bba3285fa5f0291175edc2e3a6583a82d3b2b1f14a72940
LISTING-PHY-0000009,LST-0000009,LOC-046,BRK-0017,Apartment,1,Unfurnished,1301,9395822,7222,2025-02-04,2025-06-11T17:21:51,2025-06-09,rented,BROKER_CRM,SRC-L-0000009,BATCH-2026-01,listings.parquet,/Volumes/workspace/default/propiq/listings.parquet,2026-07-30T14:04:08.257Z,W04_PROPIQ_RUN01,propiq_source_v1.0,f6053873890db17304c3ae71ff571ff8ef8cc4b9362e61a2880f67c662b0a08b
LISTING-PHY-0000010,LST-0000010,LOC-071,BRK-0201,Villa,2,Semi-furnished,3882,39615810,10205,2024-02-12,2024-04-30T07:39:38,2024-05-01,sold,PORTAL_FEED_A,SRC-L-0000010,BATCH-2026-01,listings.parquet,/Volumes/workspace/default/propiq/listings.parquet,2026-07-30T14:04:08.257Z,W04_PROPIQ_RUN01,propiq_source_v1.0,

Confirm that the approved business columns and all technical metadata columns are present.

## 3.5 Reconcile counts

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM listings_source)   AS source_count,
  (SELECT COUNT(*) FROM bronze_propiq_listings)  AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM listings_source) = (SELECT COUNT(*) FROM bronze_propiq_listings)
    THEN 'MATCH'
    ELSE 'CHECK'
  END AS reconciliation_result;


source_count,bronze_count,reconciliation_result
50200,50200,MATCH


**Checkpoint — `bronze_propiq_listings`:**
- source view opens and business columns are present;
- Bronze Delta table exists with all technical metadata columns populated;
- source and Bronze counts reconcile to `MATCH`.


# Part 4 — Build `bronze_propiq_leads`

## 4.1 Identify the source

| Item | Value |
|---|---|
| Source file | `leads.csv` |
| Format | CSV |
| Volume path | `/Volumes/workspace/default/propiq/leads.csv` |
| Target Bronze table | `bronze_propiq_leads` |



### Read the source (CSV reader)

The temp view declares every source field explicitly as `STRING`. Keeping
Bronze fields as strings avoids any accidental business-value conversion
during ingestion. `_corrupt_record` captures any row the reader could not
parse against this declared schema — that context is preserved, not
discarded.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW leads_source
(
  record_uid STRING,
  lead_id STRING,
  listing_id STRING,
  lead_channel STRING,
  buyer_intent STRING,
  qualified_flag STRING,
  lead_status STRING,
  _corrupt_record STRING
)
USING CSV
OPTIONS (
  path '/Volumes/workspace/default/propiq/leads.csv',
  header 'true',
  mode 'PERMISSIVE',
  columnNameOfCorruptRecord '_corrupt_record'
);


## 4.2 Inspect the source

In [0]:
%sql
SELECT * FROM leads_source LIMIT 10;


record_uid,lead_id,listing_id,lead_channel,buyer_intent,qualified_flag,lead_status,_corrupt_record
LEAD-PHY-0000001,LED-00000001,LST-0018000,2026-05-25T18:19:50,Campaign,Medium,False,"LEAD-PHY-0000001,LED-00000001,LST-0018000,2026-05-25T18:19:50,Campaign,Medium,False,new,2Cr-5Cr,CAMPAIGN,SRC-D-00000001,BATCH-2026-01"
LEAD-PHY-0000002,LED-00000002,LST-0024708,2025-10-22T21:10:21,Portal Search,Medium,True,"LEAD-PHY-0000002,LED-00000002,LST-0024708,2025-10-22T21:10:21,Portal Search,Medium,True,qualified,1Cr-2Cr,MOBILE,SRC-D-00000002,BATCH-2026-01"
LEAD-PHY-0000003,LED-00000003,LST-0020315,2026-04-17T01:24:48,Portal Search,High,False,"LEAD-PHY-0000003,LED-00000003,LST-0020315,2026-04-17T01:24:48,Portal Search,High,False,new,Under 50L,BROKER_CRM,SRC-D-00000003,BATCH-2026-01"
LEAD-PHY-0000004,LED-00000004,LST-0015485,2025-03-01T21:32:16,Partner,Medium,True,"LEAD-PHY-0000004,LED-00000004,LST-0015485,2025-03-01T21:32:16,Partner,Medium,True,negotiation,1Cr-2Cr,MOBILE,SRC-D-00000004,BATCH-2026-01"
LEAD-PHY-0000005,LED-00000005,LST-0037002,2025-06-18T04:22:06,Portal Search,Medium,True,"LEAD-PHY-0000005,LED-00000005,LST-0037002,2025-06-18T04:22:06,Portal Search,Medium,True,qualified,1Cr-2Cr,MOBILE,SRC-D-00000005,BATCH-2026-01"
LEAD-PHY-0000006,LED-00000006,LST-0019134,2026-02-14T18:22:41,Portal Search,Exploratory,True,"LEAD-PHY-0000006,LED-00000006,LST-0019134,2026-02-14T18:22:41,Portal Search,Exploratory,True,negotiation,50L-1Cr,BROKER_CRM,SRC-D-00000006,BATCH-2026-01"
LEAD-PHY-0000007,LED-00000007,LST-0009732,2026-05-17T07:00:28,Portal Search,Medium,True,"LEAD-PHY-0000007,LED-00000007,LST-0009732,2026-05-17T07:00:28,Portal Search,Medium,True,site_visit,1Cr-2Cr,CAMPAIGN,SRC-D-00000007,BATCH-2026-01"
LEAD-PHY-0000008,LED-00000008,LST-0004899,2026-04-18T01:45:58,Portal Search,Medium,False,"LEAD-PHY-0000008,LED-00000008,LST-0004899,2026-04-18T01:45:58,Portal Search,Medium,False,closed_unqualified,50L-1Cr,WEB,SRC-D-00000008,BATCH-2026-01"
LEAD-PHY-0000009,LED-00000009,LST-0005907,2026-02-24T22:13:22,Walk-in,Exploratory,True,"LEAD-PHY-0000009,LED-00000009,LST-0005907,2026-02-24T22:13:22,Walk-in,Exploratory,True,negotiation,50L-1Cr,WEB,SRC-D-00000009,BATCH-2026-01"
LEAD-PHY-0000010,LED-00000010,LST-0017045,2025-02-27T15:19:28,Portal Search,Exploratory,True,"LEAD-PHY-0000010,LED-00000010,LST-0017045,2025-02-27T15:19:28,Portal Search,Exploratory,True,qualified,Under 50L,BROKER_CRM,SRC-D-00000010,BATCH-2026-01"


Confirm the approved business columns are present with no unexpected merged or missing fields.

In [0]:
%sql
DESCRIBE leads_source;


col_name,data_type,comment
record_uid,string,null
lead_id,string,null
listing_id,string,null
lead_channel,string,null
buyer_intent,string,null
qualified_flag,string,null
lead_status,string,null
_corrupt_record,string,null


In [0]:
%sql
SELECT COUNT(*) AS leads_source_count
FROM leads_source;


leads_source_count
120800


Record the displayed source count in the Week-4 log before continuing.

## 4.3 Create Bronze — add metadata

`leads_bronze_ready` keeps every approved business column untouched and adds:

- `_source_file_name`, `_source_file_path` — where the row came from;
- `_ingested_at` — when the row entered Bronze;
- `_ingestion_run_id`, `_schema_version` — which controlled run and source
  contract produced this row;
- `_record_hash` — a repeatable fingerprint built from the business columns
  in a fixed order, so any future silent value change can be detected;
- `_rescued_payload` — carries forward any row the reader could not fully parse.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW leads_bronze_ready AS
SELECT
  s.record_uid,
  s.lead_id,
  s.listing_id,
  s.lead_channel,
  s.buyer_intent,
  s.qualified_flag,
  s.lead_status,
  'leads.csv' AS _source_file_name,
  '/Volumes/workspace/default/propiq/leads.csv' AS _source_file_path,
  current_timestamp() AS _ingested_at,
  r.ingestion_run_id AS _ingestion_run_id,
  r.schema_version AS _schema_version,
  s._corrupt_record AS _rescued_payload,
  sha2(concat_ws('||',
    coalesce(cast(s.record_uid AS STRING), '<NULL>'),
    coalesce(cast(s.lead_id AS STRING), '<NULL>'),
    coalesce(cast(s.listing_id AS STRING), '<NULL>'),
    coalesce(cast(s.lead_channel AS STRING), '<NULL>'),
    coalesce(cast(s.buyer_intent AS STRING), '<NULL>'),
    coalesce(cast(s.qualified_flag AS STRING), '<NULL>'),
    coalesce(cast(s.lead_status AS STRING), '<NULL>')
  ), 256) AS _record_hash
FROM leads_source s
CROSS JOIN week4_run_control r;


In [0]:
%sql
SELECT * FROM leads_bronze_ready LIMIT 10;


record_uid,lead_id,listing_id,lead_channel,buyer_intent,qualified_flag,lead_status,_source_file_name,_source_file_path,_ingested_at,_ingestion_run_id,_schema_version,_rescued_payload,_record_hash
LEAD-PHY-0000001,LED-00000001,LST-0018000,2026-05-25T18:19:50,Campaign,Medium,False,leads.csv,/Volumes/workspace/default/propiq/leads.csv,2026-07-30T14:04:29.297Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"LEAD-PHY-0000001,LED-00000001,LST-0018000,2026-05-25T18:19:50,Campaign,Medium,False,new,2Cr-5Cr,CAMPAIGN,SRC-D-00000001,BATCH-2026-01",b2d8cc328dc935c73804cb42b2287e6db0e138c285c2519902044af549f6a4a4
LEAD-PHY-0000002,LED-00000002,LST-0024708,2025-10-22T21:10:21,Portal Search,Medium,True,leads.csv,/Volumes/workspace/default/propiq/leads.csv,2026-07-30T14:04:29.297Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"LEAD-PHY-0000002,LED-00000002,LST-0024708,2025-10-22T21:10:21,Portal Search,Medium,True,qualified,1Cr-2Cr,MOBILE,SRC-D-00000002,BATCH-2026-01",e4eea99f124728568422f3398c73e0d5b0d4417bcca16d39acdb3aecbd6d461d
LEAD-PHY-0000003,LED-00000003,LST-0020315,2026-04-17T01:24:48,Portal Search,High,False,leads.csv,/Volumes/workspace/default/propiq/leads.csv,2026-07-30T14:04:29.297Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"LEAD-PHY-0000003,LED-00000003,LST-0020315,2026-04-17T01:24:48,Portal Search,High,False,new,Under 50L,BROKER_CRM,SRC-D-00000003,BATCH-2026-01",4c1bf77a6143f75c509f0079a0dbe6bfcd8863d381459a32efd867b673e48f3b
LEAD-PHY-0000004,LED-00000004,LST-0015485,2025-03-01T21:32:16,Partner,Medium,True,leads.csv,/Volumes/workspace/default/propiq/leads.csv,2026-07-30T14:04:29.297Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"LEAD-PHY-0000004,LED-00000004,LST-0015485,2025-03-01T21:32:16,Partner,Medium,True,negotiation,1Cr-2Cr,MOBILE,SRC-D-00000004,BATCH-2026-01",8a51b4f32b6f92cff8380b527dfb0d40f78d99ac2c4928bed7681a1328edb541
LEAD-PHY-0000005,LED-00000005,LST-0037002,2025-06-18T04:22:06,Portal Search,Medium,True,leads.csv,/Volumes/workspace/default/propiq/leads.csv,2026-07-30T14:04:29.297Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"LEAD-PHY-0000005,LED-00000005,LST-0037002,2025-06-18T04:22:06,Portal Search,Medium,True,qualified,1Cr-2Cr,MOBILE,SRC-D-00000005,BATCH-2026-01",05e330145724c59beb28c8794d22aee22d45c60ba4e951d86b366e0c7f5b759e
LEAD-PHY-0000006,LED-00000006,LST-0019134,2026-02-14T18:22:41,Portal Search,Exploratory,True,leads.csv,/Volumes/workspace/default/propiq/leads.csv,2026-07-30T14:04:29.297Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"LEAD-PHY-0000006,LED-00000006,LST-0019134,2026-02-14T18:22:41,Portal Search,Exploratory,True,negotiation,50L-1Cr,BROKER_CRM,SRC-D-00000006,BATCH-2026-01",aa10c8c14d9fe3512898b297733ddca4e8968d1e25b8df6536b8aafeb01d653d
LEAD-PHY-0000007,LED-00000007,LST-0009732,2026-05-17T07:00:28,Portal Search,Medium,True,leads.csv,/Volumes/workspace/default/propiq/leads.csv,2026-07-30T14:04:29.297Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"LEAD-PHY-0000007,LED-00000007,LST-0009732,2026-05-17T07:00:28,Portal Search,Medium,True,site_visit,1Cr-2Cr,CAMPAIGN,SRC-D-00000007,BATCH-2026-01",c0991e3121a148ae18003ed7e8b2710e884a2fd145ce3118547430d56643d1a9
LEAD-PHY-0000008,LED-00000008,LST-0004899,2026-04-18T01:45:58,Portal Search,Medium,False,leads.csv,/Volumes/workspace/default/propiq/leads.csv,2026-07-30T14:04:29.297Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"LEAD-PHY-0000008,LED-00000008,LST-0004899,2026-04-18T01:45:58,Portal Search,Medium,False,closed_unqualified,50L-1Cr,WEB,SRC-D-00000008,BATCH-2026-01",dcfaa583f9f5e4a1df4e31a88d64857d0ec2a66c298282b2f9103936ba58a131
LEAD-PHY-0000009,LED-00000009,LST-0005907,2026-02-24T22:13:22,Walk-in,Exploratory,True,leads.csv,/Volumes/workspace/default/propiq/leads.csv,2026-07-30T14:04:29.297Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"LEAD-PHY-0000009,LED-00000009,LST-0005907,2026-02-24T22:13:22,Walk-in,Exploratory,True,negotiation,50L-1Cr,WEB,SRC-D-00000009,BATCH-2026-01",0a9150dc91cf75c73949b88623adf720ac6ed3fdfd4bfbab875aa5548dc48ffe
LEAD-PHY-0000010,LED-00000010,LST-0017045,2025-02-27T15:19:28,Portal Search,Exploratory,True,lea

### Persist `bronze_propiq_leads` as Delta

`CREATE OR REPLACE TABLE ... USING DELTA` writes the complete controlled
snapshot as a persistent Unity Catalog Delta table, supporting the
controlled full-refresh / safe-rerun pattern used in Week 4.

In [0]:
%sql
CREATE OR REPLACE TABLE bronze_propiq_leads
USING DELTA
AS
SELECT * FROM leads_bronze_ready;


num_affected_rows,num_inserted_rows


## 4.4 Verify the table

In [0]:
%sql
DESCRIBE DETAIL bronze_propiq_leads;


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,b13d858f-aac1-410c-a241-20424c981635,workspace.default.bronze_propiq_leads,null,,2026-07-30T14:04:31.831Z,2026-07-30T14:04:35.000Z,List(),List(),1,8246311,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE TABLE bronze_propiq_leads;


col_name,data_type,comment
record_uid,string,null
lead_id,string,null
listing_id,string,null
lead_channel,string,null
buyer_intent,string,null
qualified_flag,string,null
lead_status,string,null
_source_file_name,string,null
_source_file_path,string,null
_ingested_at,timestamp,null


In [0]:
%sql
SELECT * FROM bronze_propiq_leads LIMIT 10;


record_uid,lead_id,listing_id,lead_channel,buyer_intent,qualified_flag,lead_status,_source_file_name,_source_file_path,_ingested_at,_ingestion_run_id,_schema_version,_rescued_payload,_record_hash
LEAD-PHY-0000001,LED-00000001,LST-0018000,2026-05-25T18:19:50,Campaign,Medium,False,leads.csv,/Volumes/workspace/default/propiq/leads.csv,2026-07-30T14:04:32.387Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"LEAD-PHY-0000001,LED-00000001,LST-0018000,2026-05-25T18:19:50,Campaign,Medium,False,new,2Cr-5Cr,CAMPAIGN,SRC-D-00000001,BATCH-2026-01",b2d8cc328dc935c73804cb42b2287e6db0e138c285c2519902044af549f6a4a4
LEAD-PHY-0000002,LED-00000002,LST-0024708,2025-10-22T21:10:21,Portal Search,Medium,True,leads.csv,/Volumes/workspace/default/propiq/leads.csv,2026-07-30T14:04:32.387Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"LEAD-PHY-0000002,LED-00000002,LST-0024708,2025-10-22T21:10:21,Portal Search,Medium,True,qualified,1Cr-2Cr,MOBILE,SRC-D-00000002,BATCH-2026-01",e4eea99f124728568422f3398c73e0d5b0d4417bcca16d39acdb3aecbd6d461d
LEAD-PHY-0000003,LED-00000003,LST-0020315,2026-04-17T01:24:48,Portal Search,High,False,leads.csv,/Volumes/workspace/default/propiq/leads.csv,2026-07-30T14:04:32.387Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"LEAD-PHY-0000003,LED-00000003,LST-0020315,2026-04-17T01:24:48,Portal Search,High,False,new,Under 50L,BROKER_CRM,SRC-D-00000003,BATCH-2026-01",4c1bf77a6143f75c509f0079a0dbe6bfcd8863d381459a32efd867b673e48f3b
LEAD-PHY-0000004,LED-00000004,LST-0015485,2025-03-01T21:32:16,Partner,Medium,True,leads.csv,/Volumes/workspace/default/propiq/leads.csv,2026-07-30T14:04:32.387Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"LEAD-PHY-0000004,LED-00000004,LST-0015485,2025-03-01T21:32:16,Partner,Medium,True,negotiation,1Cr-2Cr,MOBILE,SRC-D-00000004,BATCH-2026-01",8a51b4f32b6f92cff8380b527dfb0d40f78d99ac2c4928bed7681a1328edb541
LEAD-PHY-0000005,LED-00000005,LST-0037002,2025-06-18T04:22:06,Portal Search,Medium,True,leads.csv,/Volumes/workspace/default/propiq/leads.csv,2026-07-30T14:04:32.387Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"LEAD-PHY-0000005,LED-00000005,LST-0037002,2025-06-18T04:22:06,Portal Search,Medium,True,qualified,1Cr-2Cr,MOBILE,SRC-D-00000005,BATCH-2026-01",05e330145724c59beb28c8794d22aee22d45c60ba4e951d86b366e0c7f5b759e
LEAD-PHY-0000006,LED-00000006,LST-0019134,2026-02-14T18:22:41,Portal Search,Exploratory,True,leads.csv,/Volumes/workspace/default/propiq/leads.csv,2026-07-30T14:04:32.387Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"LEAD-PHY-0000006,LED-00000006,LST-0019134,2026-02-14T18:22:41,Portal Search,Exploratory,True,negotiation,50L-1Cr,BROKER_CRM,SRC-D-00000006,BATCH-2026-01",aa10c8c14d9fe3512898b297733ddca4e8968d1e25b8df6536b8aafeb01d653d
LEAD-PHY-0000007,LED-00000007,LST-0009732,2026-05-17T07:00:28,Portal Search,Medium,True,leads.csv,/Volumes/workspace/default/propiq/leads.csv,2026-07-30T14:04:32.387Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"LEAD-PHY-0000007,LED-00000007,LST-0009732,2026-05-17T07:00:28,Portal Search,Medium,True,site_visit,1Cr-2Cr,CAMPAIGN,SRC-D-00000007,BATCH-2026-01",c0991e3121a148ae18003ed7e8b2710e884a2fd145ce3118547430d56643d1a9
LEAD-PHY-0000008,LED-00000008,LST-0004899,2026-04-18T01:45:58,Portal Search,Medium,False,leads.csv,/Volumes/workspace/default/propiq/leads.csv,2026-07-30T14:04:32.387Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"LEAD-PHY-0000008,LED-00000008,LST-0004899,2026-04-18T01:45:58,Portal Search,Medium,False,closed_unqualified,50L-1Cr,WEB,SRC-D-00000008,BATCH-2026-01",dcfaa583f9f5e4a1df4e31a88d64857d0ec2a66c298282b2f9103936ba58a131
LEAD-PHY-0000009,LED-00000009,LST-0005907,2026-02-24T22:13:22,Walk-in,Exploratory,True,leads.csv,/Volumes/workspace/default/propiq/leads.csv,2026-07-30T14:04:32.387Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"LEAD-PHY-0000009,LED-00000009,LST-0005907,2026-02-24T22:13:22,Walk-in,Exploratory,True,negotiation,50L-1Cr,WEB,SRC-D-00000009,BATCH-2026-01",0a9150dc91cf75c73949b88623adf720ac6ed3fdfd4bfbab875aa5548dc48ffe
LEAD-PHY-0000010,LED-00000010,LST-0017045,2025-02-27T15:19:28,Portal Search,Exploratory,True,lea

Confirm that the approved business columns and all technical metadata columns are present.

## 4.5 Reconcile counts

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM leads_source)   AS source_count,
  (SELECT COUNT(*) FROM bronze_propiq_leads)  AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM leads_source) = (SELECT COUNT(*) FROM bronze_propiq_leads)
    THEN 'MATCH'
    ELSE 'CHECK'
  END AS reconciliation_result;


source_count,bronze_count,reconciliation_result
120800,120800,MATCH


**Checkpoint — `bronze_propiq_leads`:**
- source view opens and business columns are present;
- Bronze Delta table exists with all technical metadata columns populated;
- source and Bronze counts reconcile to `MATCH`.


# Part 5 — Build `bronze_propiq_localities`

## 5.1 Identify the source

| Item | Value |
|---|---|
| Source file | `localities.json` |
| Format | JSON Lines |
| Volume path | `/Volumes/workspace/default/propiq/localities.json` |
| Target Bronze table | `bronze_propiq_localities` |



### Read the source (JSON Lines reader)

`multiLine = false` tells Spark that every line in `localities.json` is one
complete JSON object. Fields are declared as `STRING` to avoid business-value
conversion. `_corrupt_record` preserves parser context for any line that does
not match the declared schema.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW localities_source
(
  locality_id STRING,
  locality_name STRING,
  city STRING,
  city_zone STRING,
  market_segment STRING,
  _corrupt_record STRING
)
USING JSON
OPTIONS (
  path '/Volumes/workspace/default/propiq/localities.json',
  multiLine 'false',
  mode 'PERMISSIVE',
  columnNameOfCorruptRecord '_corrupt_record'
);


## 5.2 Inspect the source

In [0]:
%sql
SELECT * FROM localities_source LIMIT 10;


locality_id,locality_name,city,city_zone,market_segment,_corrupt_record
LOC-001,Hyderabad North Cluster 01,Hyderabad,North,Affordable,null
LOC-002,Bengaluru North Cluster 01,Bengaluru,North,Mid-market,null
LOC-003,Pune North Cluster 01,Pune,North,Premium,null
LOC-004,Chennai North Cluster 01,Chennai,North,Luxury,null
LOC-005,Hyderabad South Cluster 01,Hyderabad,South,Affordable,null
LOC-006,Bengaluru South Cluster 01,Bengaluru,South,Mid-market,null
LOC-007,Pune South Cluster 01,Pune,South,Premium,null
LOC-008,Chennai South Cluster 01,Chennai,South,Luxury,null
LOC-009,Hyderabad East Cluster 01,Hyderabad,East,Affordable,null
LOC-010,Bengaluru East Cluster 01,Bengaluru,East,Mid-market,null


Confirm the approved business columns are present with no unexpected merged or missing fields.

In [0]:
%sql
DESCRIBE localities_source;


col_name,data_type,comment
locality_id,string,null
locality_name,string,null
city,string,null
city_zone,string,null
market_segment,string,null
_corrupt_record,string,null


In [0]:
%sql
SELECT COUNT(*) AS localities_source_count
FROM localities_source;


localities_source_count
80


Record the displayed source count in the Week-4 log before continuing.

## 5.3 Create Bronze — add metadata

`localities_bronze_ready` keeps every approved business column untouched and adds:

- `_source_file_name`, `_source_file_path` — where the row came from;
- `_ingested_at` — when the row entered Bronze;
- `_ingestion_run_id`, `_schema_version` — which controlled run and source
  contract produced this row;
- `_record_hash` — a repeatable fingerprint built from the business columns
  in a fixed order, so any future silent value change can be detected;
- `_rescued_payload` — carries forward any row the reader could not fully parse.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW localities_bronze_ready AS
SELECT
  s.locality_id,
  s.locality_name,
  s.city,
  s.city_zone,
  s.market_segment,
  'localities.json' AS _source_file_name,
  '/Volumes/workspace/default/propiq/localities.json' AS _source_file_path,
  current_timestamp() AS _ingested_at,
  r.ingestion_run_id AS _ingestion_run_id,
  r.schema_version AS _schema_version,
  s._corrupt_record AS _rescued_payload,
  sha2(concat_ws('||',
    coalesce(cast(s.locality_id AS STRING), '<NULL>'),
    coalesce(cast(s.locality_name AS STRING), '<NULL>'),
    coalesce(cast(s.city AS STRING), '<NULL>'),
    coalesce(cast(s.city_zone AS STRING), '<NULL>'),
    coalesce(cast(s.market_segment AS STRING), '<NULL>')
  ), 256) AS _record_hash
FROM localities_source s
CROSS JOIN week4_run_control r;


In [0]:
%sql
SELECT * FROM localities_bronze_ready LIMIT 10;


locality_id,locality_name,city,city_zone,market_segment,_source_file_name,_source_file_path,_ingested_at,_ingestion_run_id,_schema_version,_rescued_payload,_record_hash
LOC-001,Hyderabad North Cluster 01,Hyderabad,North,Affordable,localities.json,/Volumes/workspace/default/propiq/localities.json,2026-07-30T14:04:46.865Z,W04_PROPIQ_RUN01,propiq_source_v1.0,null,b5ac35733faca703bcc54ee4e8f4079261e36fc6c739d5a97b12924a658f3f10
LOC-002,Bengaluru North Cluster 01,Bengaluru,North,Mid-market,localities.json,/Volumes/workspace/default/propiq/localities.json,2026-07-30T14:04:46.865Z,W04_PROPIQ_RUN01,propiq_source_v1.0,null,7cc72e89df9dca14500f9eabed840600fc41e0c2344a56b48d9ca7bff68d62c0
LOC-003,Pune North Cluster 01,Pune,North,Premium,localities.json,/Volumes/workspace/default/propiq/localities.json,2026-07-30T14:04:46.865Z,W04_PROPIQ_RUN01,propiq_source_v1.0,null,becfc8b000c23d84170149e50003a9f564a718ea7c5296b5b2a6bfe0ca24644a
LOC-004,Chennai North Cluster 01,Chennai,North,Luxury,localities.json,/Volumes/workspace/default/propiq/localities.json,2026-07-30T14:04:46.865Z,W04_PROPIQ_RUN01,propiq_source_v1.0,null,915eba7a65a73ed8cd75019169382e6a46e402798766367e23e9b903a6a96447
LOC-005,Hyderabad South Cluster 01,Hyderabad,South,Affordable,localities.json,/Volumes/workspace/default/propiq/localities.json,2026-07-30T14:04:46.865Z,W04_PROPIQ_RUN01,propiq_source_v1.0,null,c887aff6baa28f0f8a65d3d55715bfa463143af10ad4f412d3e7f41612f4fb19
LOC-006,Bengaluru South Cluster 01,Bengaluru,South,Mid-market,localities.json,/Volumes/workspace/default/propiq/localities.json,2026-07-30T14:04:46.865Z,W04_PROPIQ_RUN01,propiq_source_v1.0,null,4a36ae7e6663b413652a73dd9e16fd44d1f67f3c930290c6e8f0a1f112dfef9a
LOC-007,Pune South Cluster 01,Pune,South,Premium,localities.json,/Volumes/workspace/default/propiq/localities.json,2026-07-30T14:04:46.865Z,W04_PROPIQ_RUN01,propiq_source_v1.0,null,173ca59c488afb06a26291b7f09dc183f254d0c5c27e8a438753ad5e6e9395e9
LOC-008,Chennai South Cluster 01,Chennai,South,Luxury,localities.json,/Volumes/workspace/default/propiq/localities.json,2026-07-30T14:04:46.865Z,W04_PROPIQ_RUN01,propiq_source_v1.0,null,a8c8e5fdccef11a5a6da3e5ba7c8e434030bb2808e150af6ceee6d6b17145c89
LOC-009,Hyderabad East Cluster 01,Hyderabad,East,Affordable,localities.json,/Volumes/workspace/default/propiq/localities.json,2026-07-30T14:04:46.865Z,W04_PROPIQ_RUN01,propiq_source_v1.0,null,4bcada4c4777dfe79203bc0907d22c51657ebe494456d45492b8c2f83a58c4d0
LOC-010,Bengaluru East Cluster 01,Bengaluru,East,Mid-market,localities.json,/Volumes/workspace/default/propiq/localities.json,2026-07-30T14:04:46.865Z,W04_PROPIQ_RUN01,propiq_source_v1.0,null,5acabf330fcce1dfb4fb37e7a937d06dec65b77026cff313947f647ba568579f


### Persist `bronze_propiq_localities` as Delta

`CREATE OR REPLACE TABLE ... USING DELTA` writes the complete controlled
snapshot as a persistent Unity Catalog Delta table, supporting the
controlled full-refresh / safe-rerun pattern used in Week 4.

In [0]:
%sql
CREATE OR REPLACE TABLE bronze_propiq_localities
USING DELTA
AS
SELECT * FROM localities_bronze_ready;


num_affected_rows,num_inserted_rows


## 5.4 Verify the table

In [0]:
%sql
DESCRIBE DETAIL bronze_propiq_localities;


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,544b21a3-acd3-40d3-818c-1334514fac81,workspace.default.bronze_propiq_localities,null,,2026-07-30T14:04:48.557Z,2026-07-30T14:04:51.000Z,List(),List(),1,7568,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE TABLE bronze_propiq_localities;


col_name,data_type,comment
locality_id,string,null
locality_name,string,null
city,string,null
city_zone,string,null
market_segment,string,null
_source_file_name,string,null
_source_file_path,string,null
_ingested_at,timestamp,null
_ingestion_run_id,string,null
_schema_version,string,null


In [0]:
%sql
SELECT * FROM bronze_propiq_localities LIMIT 10;


locality_id,locality_name,city,city_zone,market_segment,_source_file_name,_source_file_path,_ingested_at,_ingestion_run_id,_schema_version,_rescued_payload,_record_hash
LOC-001,Hyderabad North Cluster 01,Hyderabad,North,Affordable,localities.json,/Volumes/workspace/default/propiq/localities.json,2026-07-30T14:04:48.968Z,W04_PROPIQ_RUN01,propiq_source_v1.0,null,b5ac35733faca703bcc54ee4e8f4079261e36fc6c739d5a97b12924a658f3f10
LOC-002,Bengaluru North Cluster 01,Bengaluru,North,Mid-market,localities.json,/Volumes/workspace/default/propiq/localities.json,2026-07-30T14:04:48.968Z,W04_PROPIQ_RUN01,propiq_source_v1.0,null,7cc72e89df9dca14500f9eabed840600fc41e0c2344a56b48d9ca7bff68d62c0
LOC-003,Pune North Cluster 01,Pune,North,Premium,localities.json,/Volumes/workspace/default/propiq/localities.json,2026-07-30T14:04:48.968Z,W04_PROPIQ_RUN01,propiq_source_v1.0,null,becfc8b000c23d84170149e50003a9f564a718ea7c5296b5b2a6bfe0ca24644a
LOC-004,Chennai North Cluster 01,Chennai,North,Luxury,localities.json,/Volumes/workspace/default/propiq/localities.json,2026-07-30T14:04:48.968Z,W04_PROPIQ_RUN01,propiq_source_v1.0,null,915eba7a65a73ed8cd75019169382e6a46e402798766367e23e9b903a6a96447
LOC-005,Hyderabad South Cluster 01,Hyderabad,South,Affordable,localities.json,/Volumes/workspace/default/propiq/localities.json,2026-07-30T14:04:48.968Z,W04_PROPIQ_RUN01,propiq_source_v1.0,null,c887aff6baa28f0f8a65d3d55715bfa463143af10ad4f412d3e7f41612f4fb19
LOC-006,Bengaluru South Cluster 01,Bengaluru,South,Mid-market,localities.json,/Volumes/workspace/default/propiq/localities.json,2026-07-30T14:04:48.968Z,W04_PROPIQ_RUN01,propiq_source_v1.0,null,4a36ae7e6663b413652a73dd9e16fd44d1f67f3c930290c6e8f0a1f112dfef9a
LOC-007,Pune South Cluster 01,Pune,South,Premium,localities.json,/Volumes/workspace/default/propiq/localities.json,2026-07-30T14:04:48.968Z,W04_PROPIQ_RUN01,propiq_source_v1.0,null,173ca59c488afb06a26291b7f09dc183f254d0c5c27e8a438753ad5e6e9395e9
LOC-008,Chennai South Cluster 01,Chennai,South,Luxury,localities.json,/Volumes/workspace/default/propiq/localities.json,2026-07-30T14:04:48.968Z,W04_PROPIQ_RUN01,propiq_source_v1.0,null,a8c8e5fdccef11a5a6da3e5ba7c8e434030bb2808e150af6ceee6d6b17145c89
LOC-009,Hyderabad East Cluster 01,Hyderabad,East,Affordable,localities.json,/Volumes/workspace/default/propiq/localities.json,2026-07-30T14:04:48.968Z,W04_PROPIQ_RUN01,propiq_source_v1.0,null,4bcada4c4777dfe79203bc0907d22c51657ebe494456d45492b8c2f83a58c4d0
LOC-010,Bengaluru East Cluster 01,Bengaluru,East,Mid-market,localities.json,/Volumes/workspace/default/propiq/localities.json,2026-07-30T14:04:48.968Z,W04_PROPIQ_RUN01,propiq_source_v1.0,null,5acabf330fcce1dfb4fb37e7a937d06dec65b77026cff313947f647ba568579f


Confirm that the approved business columns and all technical metadata columns are present.

## 5.5 Reconcile counts

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM localities_source)   AS source_count,
  (SELECT COUNT(*) FROM bronze_propiq_localities)  AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM localities_source) = (SELECT COUNT(*) FROM bronze_propiq_localities)
    THEN 'MATCH'
    ELSE 'CHECK'
  END AS reconciliation_result;


source_count,bronze_count,reconciliation_result
80,80,MATCH


**Checkpoint — `bronze_propiq_localities`:**
- source view opens and business columns are present;
- Bronze Delta table exists with all technical metadata columns populated;
- source and Bronze counts reconcile to `MATCH`.


# Part 6 — Build `bronze_propiq_brokers`

## 6.1 Identify the source

| Item | Value |
|---|---|
| Source file | `brokers.csv` |
| Format | CSV |
| Volume path | `/Volumes/workspace/default/propiq/brokers.csv` |
| Target Bronze table | `bronze_propiq_brokers` |



### Read the source (CSV reader)

The temp view declares every source field explicitly as `STRING`. Keeping
Bronze fields as strings avoids any accidental business-value conversion
during ingestion. `_corrupt_record` captures any row the reader could not
parse against this declared schema — that context is preserved, not
discarded.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW brokers_source
(
  broker_id STRING,
  agency_name STRING,
  broker_tier STRING,
  service_rating STRING,
  _corrupt_record STRING
)
USING CSV
OPTIONS (
  path '/Volumes/workspace/default/propiq/brokers.csv',
  header 'true',
  mode 'PERMISSIVE',
  columnNameOfCorruptRecord '_corrupt_record'
);


## 6.2 Inspect the source

In [0]:
%sql
SELECT * FROM brokers_source LIMIT 10;


broker_id,agency_name,broker_tier,service_rating,_corrupt_record
BRK-0001,PropIQ Synthetic Agency 001,Hyderabad,Standard,"BRK-0001,PropIQ Synthetic Agency 001,Hyderabad,Standard,False,2021-01-01,3.0,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000001"
BRK-0002,PropIQ Synthetic Agency 002,Bengaluru,Verified,"BRK-0002,PropIQ Synthetic Agency 002,Bengaluru,Verified,True,2021-01-20,4.7,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000002"
BRK-0003,PropIQ Synthetic Agency 003,Pune,Premier,"BRK-0003,PropIQ Synthetic Agency 003,Pune,Premier,True,2021-02-08,4.4,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000003"
BRK-0004,PropIQ Synthetic Agency 004,Chennai,Standard,"BRK-0004,PropIQ Synthetic Agency 004,Chennai,Standard,True,2021-02-27,4.1,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000004"
BRK-0005,PropIQ Synthetic Agency 005,Hyderabad,Verified,"BRK-0005,PropIQ Synthetic Agency 005,Hyderabad,Verified,True,2021-03-18,3.8,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000005"
BRK-0006,PropIQ Synthetic Agency 006,Bengaluru,Premier,"BRK-0006,PropIQ Synthetic Agency 006,Bengaluru,Premier,True,2021-04-06,3.5,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000006"
BRK-0007,PropIQ Synthetic Agency 007,Pune,Standard,"BRK-0007,PropIQ Synthetic Agency 007,Pune,Standard,True,2021-04-25,3.2,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000007"
BRK-0008,PropIQ Synthetic Agency 008,Chennai,Verified,"BRK-0008,PropIQ Synthetic Agency 008,Chennai,Verified,True,2021-05-14,4.9,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000008"
BRK-0009,PropIQ Synthetic Agency 009,Hyderabad,Premier,"BRK-0009,PropIQ Synthetic Agency 009,Hyderabad,Premier,True,2021-06-02,4.6,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000009"
BRK-0010,PropIQ Synthetic Agency 010,Bengaluru,Standard,"BRK-0010,PropIQ Synthetic Agency 010,Bengaluru,Standard,True,2021-06-21,4.3,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000010"


Confirm the approved business columns are present with no unexpected merged or missing fields.

In [0]:
%sql
DESCRIBE brokers_source;


col_name,data_type,comment
broker_id,string,null
agency_name,string,null
broker_tier,string,null
service_rating,string,null
_corrupt_record,string,null


In [0]:
%sql
SELECT COUNT(*) AS brokers_source_count
FROM brokers_source;


brokers_source_count
320


Record the displayed source count in the Week-4 log before continuing.

## 6.3 Create Bronze — add metadata

`brokers_bronze_ready` keeps every approved business column untouched and adds:

- `_source_file_name`, `_source_file_path` — where the row came from;
- `_ingested_at` — when the row entered Bronze;
- `_ingestion_run_id`, `_schema_version` — which controlled run and source
  contract produced this row;
- `_record_hash` — a repeatable fingerprint built from the business columns
  in a fixed order, so any future silent value change can be detected;
- `_rescued_payload` — carries forward any row the reader could not fully parse.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW brokers_bronze_ready AS
SELECT
  s.broker_id,
  s.agency_name,
  s.broker_tier,
  s.service_rating,
  'brokers.csv' AS _source_file_name,
  '/Volumes/workspace/default/propiq/brokers.csv' AS _source_file_path,
  current_timestamp() AS _ingested_at,
  r.ingestion_run_id AS _ingestion_run_id,
  r.schema_version AS _schema_version,
  s._corrupt_record AS _rescued_payload,
  sha2(concat_ws('||',
    coalesce(cast(s.broker_id AS STRING), '<NULL>'),
    coalesce(cast(s.agency_name AS STRING), '<NULL>'),
    coalesce(cast(s.broker_tier AS STRING), '<NULL>'),
    coalesce(cast(s.service_rating AS STRING), '<NULL>')
  ), 256) AS _record_hash
FROM brokers_source s
CROSS JOIN week4_run_control r;


In [0]:
%sql
SELECT * FROM brokers_bronze_ready LIMIT 10;


broker_id,agency_name,broker_tier,service_rating,_source_file_name,_source_file_path,_ingested_at,_ingestion_run_id,_schema_version,_rescued_payload,_record_hash
BRK-0001,PropIQ Synthetic Agency 001,Hyderabad,Standard,brokers.csv,/Volumes/workspace/default/propiq/brokers.csv,2026-07-30T14:05:01.158Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"BRK-0001,PropIQ Synthetic Agency 001,Hyderabad,Standard,False,2021-01-01,3.0,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000001",8d993826174bbd54ed2a65635001735b5fe12573b61e53edbea6845b2e6d397a
BRK-0002,PropIQ Synthetic Agency 002,Bengaluru,Verified,brokers.csv,/Volumes/workspace/default/propiq/brokers.csv,2026-07-30T14:05:01.158Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"BRK-0002,PropIQ Synthetic Agency 002,Bengaluru,Verified,True,2021-01-20,4.7,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000002",31d5de9d0566d4ca114ee4edac1fed7f241442f1062314a5df6357054fa0956f
BRK-0003,PropIQ Synthetic Agency 003,Pune,Premier,brokers.csv,/Volumes/workspace/default/propiq/brokers.csv,2026-07-30T14:05:01.158Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"BRK-0003,PropIQ Synthetic Agency 003,Pune,Premier,True,2021-02-08,4.4,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000003",b6feb30e9783099c7054408fbdbed8abb9a93f79205643367d94fb2d9a08f08e
BRK-0004,PropIQ Synthetic Agency 004,Chennai,Standard,brokers.csv,/Volumes/workspace/default/propiq/brokers.csv,2026-07-30T14:05:01.158Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"BRK-0004,PropIQ Synthetic Agency 004,Chennai,Standard,True,2021-02-27,4.1,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000004",08771c89525ef1742398ba7ea70679e33fdf9b0616e7bded60f706eb508699fa
BRK-0005,PropIQ Synthetic Agency 005,Hyderabad,Verified,brokers.csv,/Volumes/workspace/default/propiq/brokers.csv,2026-07-30T14:05:01.158Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"BRK-0005,PropIQ Synthetic Agency 005,Hyderabad,Verified,True,2021-03-18,3.8,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000005",5ec6e0449aa55941f76eddeb185d844cfadcdb3ab1e43344c50e8bc1ddec016f
BRK-0006,PropIQ Synthetic Agency 006,Bengaluru,Premier,brokers.csv,/Volumes/workspace/default/propiq/brokers.csv,2026-07-30T14:05:01.158Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"BRK-0006,PropIQ Synthetic Agency 006,Bengaluru,Premier,True,2021-04-06,3.5,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000006",6cd0899b3754c20b115141b48018b1d4f5170344a084f922a93d958e16165cd8
BRK-0007,PropIQ Synthetic Agency 007,Pune,Standard,brokers.csv,/Volumes/workspace/default/propiq/brokers.csv,2026-07-30T14:05:01.158Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"BRK-0007,PropIQ Synthetic Agency 007,Pune,Standard,True,2021-04-25,3.2,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000007",df439fc6ef8281978dbc3769545da6a9ce8e09afbcf204ba92593c4d0d62d770
BRK-0008,PropIQ Synthetic Agency 008,Chennai,Verified,brokers.csv,/Volumes/workspace/default/propiq/brokers.csv,2026-07-30T14:05:01.158Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"BRK-0008,PropIQ Synthetic Agency 008,Chennai,Verified,True,2021-05-14,4.9,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000008",71207ca547064c4bc61cce2d8326db22fdeb5dcf498086bc09c67b907e46f3c7
BRK-0009,PropIQ Synthetic Agency 009,Hyderabad,Premier,brokers.csv,/Volumes/workspace/default/propiq/brokers.csv,2026-07-30T14:05:01.158Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"BRK-0009,PropIQ Synthetic Agency 009,Hyderabad,Premier,True,2021-06-02,4.6,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000009",c970b931650f93364f44994347e420bffef2b1249d4de370c00540d6c913f99f
BRK-0010,PropIQ Synthetic Agency 010,Bengaluru,Standard,brokers.csv,/Volumes/workspace/default/propiq/brokers.csv,2026-07-30T14:05:01.158Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"BRK-0010,PropIQ Synthetic Agency 010,Bengaluru,Standard,True,2021-06-21,4.3,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000010",6db796efb0ab061832dffcd7a9a4651145884e058515e2b338d8a5f08dd5f183


### Persist `bronze_propiq_brokers` as Delta

`CREATE OR REPLACE TABLE ... USING DELTA` writes the complete controlled
snapshot as a persistent Unity Catalog Delta table, supporting the
controlled full-refresh / safe-rerun pattern used in Week 4.

In [0]:
%sql
CREATE OR REPLACE TABLE bronze_propiq_brokers
USING DELTA
AS
SELECT * FROM brokers_bronze_ready;


num_affected_rows,num_inserted_rows


## 6.4 Verify the table

In [0]:
%sql
DESCRIBE DETAIL bronze_propiq_brokers;


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,57573bf9-545f-4389-9809-82b10ef6dc8c,workspace.default.bronze_propiq_brokers,null,,2026-07-30T14:05:02.957Z,2026-07-30T14:05:05.000Z,List(),List(),1,19759,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE TABLE bronze_propiq_brokers;


col_name,data_type,comment
broker_id,string,null
agency_name,string,null
broker_tier,string,null
service_rating,string,null
_source_file_name,string,null
_source_file_path,string,null
_ingested_at,timestamp,null
_ingestion_run_id,string,null
_schema_version,string,null
_rescued_payload,string,null


In [0]:
%sql
SELECT * FROM bronze_propiq_brokers LIMIT 10;


broker_id,agency_name,broker_tier,service_rating,_source_file_name,_source_file_path,_ingested_at,_ingestion_run_id,_schema_version,_rescued_payload,_record_hash
BRK-0001,PropIQ Synthetic Agency 001,Hyderabad,Standard,brokers.csv,/Volumes/workspace/default/propiq/brokers.csv,2026-07-30T14:05:03.370Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"BRK-0001,PropIQ Synthetic Agency 001,Hyderabad,Standard,False,2021-01-01,3.0,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000001",8d993826174bbd54ed2a65635001735b5fe12573b61e53edbea6845b2e6d397a
BRK-0002,PropIQ Synthetic Agency 002,Bengaluru,Verified,brokers.csv,/Volumes/workspace/default/propiq/brokers.csv,2026-07-30T14:05:03.370Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"BRK-0002,PropIQ Synthetic Agency 002,Bengaluru,Verified,True,2021-01-20,4.7,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000002",31d5de9d0566d4ca114ee4edac1fed7f241442f1062314a5df6357054fa0956f
BRK-0003,PropIQ Synthetic Agency 003,Pune,Premier,brokers.csv,/Volumes/workspace/default/propiq/brokers.csv,2026-07-30T14:05:03.370Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"BRK-0003,PropIQ Synthetic Agency 003,Pune,Premier,True,2021-02-08,4.4,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000003",b6feb30e9783099c7054408fbdbed8abb9a93f79205643367d94fb2d9a08f08e
BRK-0004,PropIQ Synthetic Agency 004,Chennai,Standard,brokers.csv,/Volumes/workspace/default/propiq/brokers.csv,2026-07-30T14:05:03.370Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"BRK-0004,PropIQ Synthetic Agency 004,Chennai,Standard,True,2021-02-27,4.1,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000004",08771c89525ef1742398ba7ea70679e33fdf9b0616e7bded60f706eb508699fa
BRK-0005,PropIQ Synthetic Agency 005,Hyderabad,Verified,brokers.csv,/Volumes/workspace/default/propiq/brokers.csv,2026-07-30T14:05:03.370Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"BRK-0005,PropIQ Synthetic Agency 005,Hyderabad,Verified,True,2021-03-18,3.8,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000005",5ec6e0449aa55941f76eddeb185d844cfadcdb3ab1e43344c50e8bc1ddec016f
BRK-0006,PropIQ Synthetic Agency 006,Bengaluru,Premier,brokers.csv,/Volumes/workspace/default/propiq/brokers.csv,2026-07-30T14:05:03.370Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"BRK-0006,PropIQ Synthetic Agency 006,Bengaluru,Premier,True,2021-04-06,3.5,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000006",6cd0899b3754c20b115141b48018b1d4f5170344a084f922a93d958e16165cd8
BRK-0007,PropIQ Synthetic Agency 007,Pune,Standard,brokers.csv,/Volumes/workspace/default/propiq/brokers.csv,2026-07-30T14:05:03.370Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"BRK-0007,PropIQ Synthetic Agency 007,Pune,Standard,True,2021-04-25,3.2,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000007",df439fc6ef8281978dbc3769545da6a9ce8e09afbcf204ba92593c4d0d62d770
BRK-0008,PropIQ Synthetic Agency 008,Chennai,Verified,brokers.csv,/Volumes/workspace/default/propiq/brokers.csv,2026-07-30T14:05:03.370Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"BRK-0008,PropIQ Synthetic Agency 008,Chennai,Verified,True,2021-05-14,4.9,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000008",71207ca547064c4bc61cce2d8326db22fdeb5dcf498086bc09c67b907e46f3c7
BRK-0009,PropIQ Synthetic Agency 009,Hyderabad,Premier,brokers.csv,/Volumes/workspace/default/propiq/brokers.csv,2026-07-30T14:05:03.370Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"BRK-0009,PropIQ Synthetic Agency 009,Hyderabad,Premier,True,2021-06-02,4.6,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000009",c970b931650f93364f44994347e420bffef2b1249d4de370c00540d6c913f99f
BRK-0010,PropIQ Synthetic Agency 010,Bengaluru,Standard,brokers.csv,/Volumes/workspace/default/propiq/brokers.csv,2026-07-30T14:05:03.370Z,W04_PROPIQ_RUN01,propiq_source_v1.0,"BRK-0010,PropIQ Synthetic Agency 010,Bengaluru,Standard,True,2021-06-21,4.3,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000010",6db796efb0ab061832dffcd7a9a4651145884e058515e2b338d8a5f08dd5f183


Confirm that the approved business columns and all technical metadata columns are present.

## 6.5 Reconcile counts

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM brokers_source)   AS source_count,
  (SELECT COUNT(*) FROM bronze_propiq_brokers)  AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM brokers_source) = (SELECT COUNT(*) FROM bronze_propiq_brokers)
    THEN 'MATCH'
    ELSE 'CHECK'
  END AS reconciliation_result;


source_count,bronze_count,reconciliation_result
320,320,MATCH


**Checkpoint — `bronze_propiq_brokers`:**
- source view opens and business columns are present;
- Bronze Delta table exists with all technical metadata columns populated;
- source and Bronze counts reconcile to `MATCH`.


# Part 7 — Validate the complete Week-4 Bronze load

All four approved batch sources now have persistent Bronze Delta tables.
This part validates them as one controlled Week-4 load, per Section 6 of the
Week-4 Student Guide.

## 7.1 Confirm all four Bronze tables are registered

In [0]:
%sql
SHOW TABLES LIKE 'bronze_propiq_*';


database,tableName,isTemporary
default,bronze_propiq_brokers,false
default,bronze_propiq_leads,false
default,bronze_propiq_listings,false
default,bronze_propiq_localities,false


**Expected observation:** the result includes `bronze_propiq_listings`,
`bronze_propiq_leads`, `bronze_propiq_localities` and `bronze_propiq_brokers`.
If one is missing, return to that source's section before continuing.

## 7.2 One consolidated reconciliation result

In [0]:
%sql
WITH counts AS (
SELECT 'listings' AS dataset,
       (SELECT COUNT(*) FROM listings_source) AS source_count,
       (SELECT COUNT(*) FROM bronze_propiq_listings) AS bronze_count
UNION ALL
SELECT 'leads' AS dataset,
       (SELECT COUNT(*) FROM leads_source) AS source_count,
       (SELECT COUNT(*) FROM bronze_propiq_leads) AS bronze_count
UNION ALL
SELECT 'localities' AS dataset,
       (SELECT COUNT(*) FROM localities_source) AS source_count,
       (SELECT COUNT(*) FROM bronze_propiq_localities) AS bronze_count
UNION ALL
SELECT 'brokers' AS dataset,
       (SELECT COUNT(*) FROM brokers_source) AS source_count,
       (SELECT COUNT(*) FROM bronze_propiq_brokers) AS bronze_count
)
SELECT
  dataset,
  source_count,
  bronze_count,
  CASE WHEN source_count = bronze_count THEN 'MATCH' ELSE 'CHECK' END AS reconciliation_result
FROM counts
ORDER BY dataset;


dataset,source_count,bronze_count,reconciliation_result
brokers,320,320,MATCH
leads,120800,120800,MATCH
listings,50200,50200,MATCH
localities,80,80,MATCH


All four rows must show `MATCH`. A `CHECK` result does not automatically
mean the source is wrong — it means the team must stop, inspect the path,
reader options and Bronze-ready view for that source, and record the genuine
cause in the Week-4 log rather than silently rerunning.

## 7.3 Metadata completeness check

In [0]:
%sql
SELECT 'listings' AS dataset,
       SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END) AS missing_file,
       SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END) AS missing_time,
       SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END) AS missing_run,
       SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END) AS missing_hash
FROM bronze_propiq_listings
UNION ALL
SELECT 'leads' AS dataset,
       SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END) AS missing_file,
       SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END) AS missing_time,
       SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END) AS missing_run,
       SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END) AS missing_hash
FROM bronze_propiq_leads
UNION ALL
SELECT 'localities' AS dataset,
       SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END) AS missing_file,
       SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END) AS missing_time,
       SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END) AS missing_run,
       SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END) AS missing_hash
FROM bronze_propiq_localities
UNION ALL
SELECT 'brokers' AS dataset,
       SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END) AS missing_file,
       SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END) AS missing_time,
       SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END) AS missing_run,
       SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END) AS missing_hash
FROM bronze_propiq_brokers;


dataset,missing_file,missing_time,missing_run,missing_hash
listings,0,0,0,0
leads,0,0,0,0
localities,0,0,0,0
brokers,0,0,0,0


Every count above should be zero. Investigate any non-zero result before capturing evidence.

## 7.4 Rerun proof — safe repeat-run behaviour

Rerun **one** complete source section without changing the source file or
the run-control values, then compare the before/after Bronze counts. The
smallest source, `brokers.csv`, is used here for the rerun proof.

**Before rerunning**, record the current row count:

In [0]:
%sql
SELECT COUNT(*) AS brokers_before_rerun
FROM bronze_propiq_brokers;


brokers_before_rerun
320


Now scroll back to **Part 6.3** and rerun the `CREATE OR REPLACE TABLE
bronze_propiq_brokers ...` cell exactly as written, without editing the
source file, path, or run-control values. Then run the next cell.

In [0]:
%sql
SELECT COUNT(*) AS brokers_after_rerun
FROM bronze_propiq_brokers;


brokers_after_rerun
320


**Expected observation:** `brokers_before_rerun` and
`brokers_after_rerun` must be equal. `CREATE OR REPLACE TABLE` performs a
controlled full refresh rather than an append, so rerunning the same
unchanged batch does not create duplicate rows.

## 7.5 Delta table details / history for the rerun-tested table

In [0]:
%sql
DESCRIBE DETAIL bronze_propiq_brokers;


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,57573bf9-545f-4389-9809-82b10ef6dc8c,workspace.default.bronze_propiq_brokers,null,,2026-07-30T14:05:02.957Z,2026-07-30T14:05:05.000Z,List(),List(),1,19759,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE HISTORY bronze_propiq_brokers;


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-07-30T14:05:05.000Z,71736716437240,thota.madhulika05@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2115632319546168),769ca35b-8026-4a6e-ab08-1a763237e5c4,0730-140317-dhozcp1w-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 320, numOutputBytes -> 19759)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


The history should show at least two `CREATE OR REPLACE TABLE AS SELECT`
operations (the original build and the rerun), each with its own version
number and timestamp — this is the audit trail that proves safe repeat-run
behaviour.

## 7.6 Confirm visibility in Catalog Explorer

In the Databricks workspace:

1. Open **Catalog** → `workspace` → `default`.
2. Confirm `bronze_propiq_listings`, `bronze_propiq_leads`,
   `bronze_propiq_localities` and `bronze_propiq_brokers` are all listed as
   tables.
3. Open each table's **Details** tab and confirm the format is `DELTA`.
4. Open each table's **Schema** tab and confirm the business columns plus
   the six/seven metadata columns are present.


# Evidence to capture (`evidence/week_04/`)

| Evidence | What to capture |
|---|---|
| W04-E01 | Volume listing showing all 4 approved batch files (Part 1) |
| W04-E02 | `bronze_propiq_listings` schema / table details |
| W04-E03 | `bronze_propiq_leads` schema / table details |
| W04-E04 | `bronze_propiq_localities` schema / table details |
| W04-E05 | `bronze_propiq_brokers` schema / table details |
| W04-E06 | Consolidated reconciliation result (Part 7.2), all rows `MATCH` |
| W04-E07 | Rerun proof — before/after counts (Part 7.4) |
| W04-E08 | `DESCRIBE HISTORY bronze_propiq_brokers` output (Part 7.5) |
| W04-E09 | Catalog Explorer screenshot showing all 4 Bronze tables |

Also update, in the repository:

- `weekly/week_04_log.md` — what was completed, blockers, corrections, each
  student's contribution;
- AI Transparency Note — where AI was used (structuring this notebook from
  the Week-4 Student Guide and the project's own Data Dictionary/manifest)
  and what the team verified manually by executing every cell in Databricks
  and recording the actual results.


# Week-4 exit checklist

- [ ] All 4 approved batch source files are covered; streaming drops are excluded.
- [ ] The correct reader is used for each file format (Parquet / CSV / JSON Lines).
- [ ] Source business values are preserved without cleansing or transformation.
- [ ] One persistent Bronze Delta table exists for every approved batch source.
- [ ] Required ingestion and lineage metadata is present on every table.
- [ ] Source and Bronze counts are reconciled (`MATCH`).
- [ ] Safe repeat-run behaviour is demonstrated with before/after counts and Delta history.
- [ ] Only real, executed evidence is included — no fabricated counts.
- [ ] Notebook, evidence, Week Log and AI Transparency Note are saved in GitHub.
- [ ] No Silver, quarantine, Gold, Power BI, Auto Loader or streaming work is included.


# Viva preparation

- **What is a Bronze table, and why do we preserve original source business
  values?** Bronze is the first persistent copy of an approved source. It
  preserves every business value untouched so later Silver/DQ work always
  has an unmodified record to trace back to.
- **Which batch source files did PropIQ ingest, and why were any files
  excluded?** `listings.parquet`, `leads.csv`, `localities.json`,
  `brokers.csv`. The `listing_status_event_drop_*.json` files were excluded
  because they are streaming-simulation drops reserved for Week 10.
- **Which lineage metadata did you add and what does each field prove?**
  `_source_file_name`/`_source_file_path` prove where a row came from;
  `_ingested_at` proves when; `_ingestion_run_id`/`_schema_version` prove
  which controlled run and contract produced the row; `_record_hash` proves
  the business content used to build the row; `_rescued_payload` (CSV/JSON
  only) preserves any unparsed content.
- **How did you verify that source and Bronze counts match?** By
  independently counting the temporary source view and the persistent
  Bronze table in the same query and comparing them, both per-source and in
  one consolidated summary.
- **What happened when you reran one source section, and how did you
  confirm that duplicates were not added?** The `brokers` section was
  rerun without changing the source file or run-control values. The
  before/after row counts stayed equal, and `DESCRIBE HISTORY` shows the
  rerun as a separate, controlled `CREATE OR REPLACE TABLE` operation rather
  than an appended duplicate batch.


# Stop here — Week 4 complete

```text
Approved batch sources (listings, leads, localities, brokers)
        -> validated source views
        -> Bronze-ready views with ingestion/lineage metadata
        -> Bronze Delta tables
        -> reconciliation (MATCH) and rerun proof
        -> Catalog Explorer + GitHub evidence
```

Data cleansing, DQ routing, quarantine, Silver transformations, Gold tables,
Power BI and streaming belong to later weeks and are out of scope here.


In [0]:
%sql
SHOW TABLES LIKE 'bronze_propiq_%';

database,tableName,isTemporary
